In [169]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import ast

%matplotlib inline

In [170]:
df = pd.read_csv('E_Clash_Royale_Cards.csv')

In [171]:
df.shape

(120, 13)

In [172]:
df.size

1560

In [173]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 120 entries, 0 to 119
Data columns (total 13 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   name               120 non-null    str    
 1   maxEvolutionLevel  38 non-null     float64
 2   elixirCost         119 non-null    float64
 3   iconUrls           120 non-null    str    
 4   evolutionIcons     38 non-null     str    
 5   rarity             120 non-null    str    
 6   type               120 non-null    str    
 7   mobility           120 non-null    str    
 8   targets            120 non-null    str    
 9   attack_type        119 non-null    str    
 10  groupCard          120 non-null    bool   
 11  usage              119 non-null    float64
 12  hitpoints          101 non-null    float64
dtypes: bool(1), float64(4), str(8)
memory usage: 11.5 KB


In [174]:
df.isnull().sum()

name                  0
maxEvolutionLevel    82
elixirCost            1
iconUrls              0
evolutionIcons       82
rarity                0
type                  0
mobility              0
targets               0
attack_type           1
groupCard             0
usage                 1
hitpoints            19
dtype: int64

In [175]:
df.head()

,name,maxEvolutionLevel,elixirCost,iconUrls,evolutionIcons,rarity,type,mobility,targets,attack_type,groupCard,usage,hitpoints
0,Archer Queen,NaN,5.0,https://api-assets.clashroyale.com/cards/300/p...,NaN,champion,troop,ground,ground,single,False,6.48,1000.0
1,Archers,1.0,3.0,https://api-assets.clashroyale.com/cards/300/W...,https://api-assets.clashroyale.com/cardevoluti...,common,troop,ground,both,single,True,1.44,304.0
2,Arrows,NaN,3.0,https://api-assets.clashroyale.com/cards/300/F...,NaN,common,spell,flying,both,splash,False,30.64,NaN
3,Baby Dragon,1.0,4.0,https://api-assets.clashroyale.com/cards/300/c...,https://api-assets.clashroyale.com/cardevoluti...,epic,troop,flying,both,splash,False,4.64,1152.0
4,Balloon,NaN,5.0,https://api-assets.clashroyale.com/cards/300/q...,NaN,epic,troop,flying,buildings,single,False,7.52,1679.0


In [176]:
print(df.columns.tolist())

['name', 'maxEvolutionLevel', 'elixirCost', 'iconUrls', 'evolutionIcons', 'rarity', 'type', 'mobility', 'targets', 'attack_type', 'groupCard', 'usage', 'hitpoints']


In [177]:
cols = ['iconUrls', 'evolutionIcons']

df.drop(cols, axis = 1, inplace = True)

In [178]:
df.head()

,name,maxEvolutionLevel,elixirCost,rarity,type,mobility,targets,attack_type,groupCard,usage,hitpoints
0,Archer Queen,NaN,5.0,champion,troop,ground,ground,single,False,6.48,1000.0
1,Archers,1.0,3.0,common,troop,ground,both,single,True,1.44,304.0
2,Arrows,NaN,3.0,common,spell,flying,both,splash,False,30.64,NaN
3,Baby Dragon,1.0,4.0,epic,troop,flying,both,splash,False,4.64,1152.0
4,Balloon,NaN,5.0,epic,troop,flying,buildings,single,False,7.52,1679.0


In [179]:
df.isnull().sum()

name                  0
maxEvolutionLevel    82
elixirCost            1
rarity                0
type                  0
mobility              0
targets               0
attack_type           1
groupCard             0
usage                 1
hitpoints            19
dtype: int64

In [180]:
# Fill missing data in numeric columns with the median (middle value).
# This is ideal for skewed data as it is not affected by extreme outliers.
df['maxEvolutionLevel'] = df['maxEvolutionLevel'].fillna(0)
df['elixirCost'] = df['elixirCost'].fillna(df['elixirCost'].median())
df['usage'] = df['usage'].fillna(df['usage'].median())


# Fill missing data in the text column with the mode (most frequent value). 
# Is required because .mode() returns a Series, and extracts the top value.
df['attack_type'] = df['attack_type'].fillna(df['attack_type'].mode()[0])

In [181]:
# NaN in hitpoints means "not applicable" (spells don't have HP), not a data gap.
# So instead of filling it with a fake value, we create a separate flag column
# to mark which rows actually have hitpoints — troops/buildings = True, spells = False.

# WRONG - this OVERWRITES the real hitpoints numbers with True/False, losing the data
# df['hitpoints'] = df['hitpoints'].notna()

# CORRECT - creates a brand new column, original hitpoints values stay untouched
df['has_hitpoints'] = df['hitpoints'].notna()

# Now you have both:
# - hitpoints: the real HP values (with NaN for spells, left as-is)
# - has_hitpoints: True/False flag you can filter on later
# e.g. df[df['has_hitpoints']] gives you only cards with real HP (excludes spells)

In [182]:
df.head()

,name,maxEvolutionLevel,elixirCost,rarity,type,mobility,targets,attack_type,groupCard,usage,hitpoints,has_hitpoints
0,Archer Queen,0.0,5.0,champion,troop,ground,ground,single,False,6.48,1000.0,True
1,Archers,1.0,3.0,common,troop,ground,both,single,True,1.44,304.0,True
2,Arrows,0.0,3.0,common,spell,flying,both,splash,False,30.64,NaN,False
3,Baby Dragon,1.0,4.0,epic,troop,flying,both,splash,False,4.64,1152.0,True
4,Balloon,0.0,5.0,epic,troop,flying,buildings,single,False,7.52,1679.0,True


In [183]:
df.isnull().sum()

name                  0
maxEvolutionLevel     0
elixirCost            0
rarity                0
type                  0
mobility              0
targets               0
attack_type           0
groupCard             0
usage                 0
hitpoints            19
has_hitpoints         0
dtype: int64

##### **Basic filtering & sorting**

###### ** 1. Find all cards with elixirCost greater than 5. **

In [184]:
df.head()

,name,maxEvolutionLevel,elixirCost,rarity,type,mobility,targets,attack_type,groupCard,usage,hitpoints,has_hitpoints
0,Archer Queen,0.0,5.0,champion,troop,ground,ground,single,False,6.48,1000.0,True
1,Archers,1.0,3.0,common,troop,ground,both,single,True,1.44,304.0,True
2,Arrows,0.0,3.0,common,spell,flying,both,splash,False,30.64,NaN,False
3,Baby Dragon,1.0,4.0,epic,troop,flying,both,splash,False,4.64,1152.0,True
4,Balloon,0.0,5.0,epic,troop,flying,buildings,single,False,7.52,1679.0,True


In [185]:
higher_elixir = df[df['elixirCost'] > 5]

# First df checks row by row if this elixirCost greater than 5?
# The result is just a list of answers: [False, False, True, False, True]

# Go back to the original stack (df), look at my list of answers.
# Pull out the cards that matched.

higher_elixir[['name','elixirCost',]]

,name,elixirCost
7,Barbarian Hut,6.0
15,Boss Bandit,6.0
24,Electro Giant,7.0
27,Elite Barbarians,6.0
28,Elixir Collector,6.0
39,Giant Skeleton,6.0
47,Goblin Giant,6.0
53,Golem,8.0
65,Lava Hound,7.0
66,Lightning,6.0


###### ** 2. Find the top 5 cards by `hitpoints`. **

In [186]:
df.head()

,name,maxEvolutionLevel,elixirCost,rarity,type,mobility,targets,attack_type,groupCard,usage,hitpoints,has_hitpoints
0,Archer Queen,0.0,5.0,champion,troop,ground,ground,single,False,6.48,1000.0,True
1,Archers,1.0,3.0,common,troop,ground,both,single,True,1.44,304.0,True
2,Arrows,0.0,3.0,common,spell,flying,both,splash,False,30.64,NaN,False
3,Baby Dragon,1.0,4.0,epic,troop,flying,both,splash,False,4.64,1152.0,True
4,Balloon,0.0,5.0,epic,troop,flying,buildings,single,False,7.52,1679.0,True


In [ ]:
top5_hitpoints = df.sort_values('hitpoints', ascending = False).head(5)
# sort_values('hitpoints', ascending=False): Rearranges your rows from the highest hitpoints to lowest.
# .head(5): Grabs only the top 5 rows from that sorted list.
# top5_hitpoints =: Saves these top 5 records into a new, smaller DataFrame.

top5_hitpoints[['name','hitpoints']]
# [['name', 'hitpoints']]: Hides all other columns (like usage or elixir cost) and 
# displays only the identity (name) and health (hitpoints) of those top 5 cards.

,name,hitpoints
53,Golem,5120.0
38,Giant,4090.0
70,Mega Knight,3993.0
24,Electro Giant,3855.0
83,P.E.K.K.A,3760.0


In [ ]:

###### 2. Find the top 5 cards by `hitpoints`.
###### 3. Find all cards of `rarity == "Legendary"` and display only `name`, `elixirCost`, `usage`.

# **Grouping & aggregation**
###### 4. What is the average `elixirCost` for each `rarity`?
###### 5. What is the average `usage` for each `type` (Troop/Spell/Building)?
###### 6. Which `rarity` has the highest average `hitpoints`?

# **Value counts / categorical**
###### 7. Count how many cards belong to each `rarity`.
###### 8. Count how many cards are `groupCard == True` vs `False`.

# **Visualization (matplotlib/seaborn)**
###### 8. Plot a bar chart of average `elixirCost` by `rarity`.
###### 10. Plot a scatter plot of `elixirCost` vs `usage`, colored by `rarity`.

# **Bonus (slightly harder)**
###### 11. Handle missing `hitpoints` values (e.g., for Spells) — decide whether to fill or drop, and justify.
###### 12. Create a new column `elixir_efficiency = hitpoints / elixirCost` and find the top 5 cards by this metric.
###### 13. Find the correlation between `elixirCost`, `usage`, and `hitpoints` using `.corr()`, and visualize it with a heatmap.
